# Predicción del crimen - Modelo para predecir la ubicación del próximo crimen

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Data exploratory (Explorar datos)
### Is it possible? 
What data is available? Is this data directly relevant to the problem? What ML method(s) should we try? 
What metric should be used to measure “success”? 

In [ ]:
cluster = pd.read_csv("cluster_location-co.csv")

crime = pd.read_csv('crime-colombia.csv', header=0)
print(crime.head())

In [ ]:
def stateToNumber(s):
    l = 0
    for x in s:
        l = l + int(hex(ord(x)),16)
    return l

xref = pd.DataFrame(crime['City'])
crime['City']=crime['City'].apply(lambda x: stateToNumber(x))
crime.head()

In [ ]:
%matplotlib inline

sns.set_style('whitegrid')
sns.countplot(x='crimeCluster',data=crime, palette='RdBu_r');

In [ ]:
%run 'graph-util/neptune.py'
%run 'graph-util/visualisation.py'

g = neptune.graphTraversal(neptune_endpoint="crime.cw8vc2udxxa2.us-east-1.neptune.amazonaws.com", 
                           neptune_port=8182)

In [ ]:
user = 'Terry'

recommendations = (g.V().hasLabel('PERSON').has('name', user).as_('person').  
  both('FRIEND').aggregate('friends').  
  both('FRIEND').
    where(P.neq('person')).where(P.without('friends')).  
  groupCount().by('name').  
  order(Scope.local).by(Column.values, Order.decr).
  next())

for key in recommendations.keys():
    print(key + ': ' + str(recommendations[key]))

In [ ]:
results = g.V().hasLabel('PERSON').has('name', user).as_('person'). \
  both('FRIEND').aggregate('friends'). \
  both('FRIEND'). \
    where(P.neq('person')).where(P.without('friends')).aggregate('not-friends').by('name'). \
  path().by('name').aggregate('p'). \
  project('not-friends', 'paths'). \
  by(__.select('not-friends').unfold().dedup().fold()). \
  by(__.select('p')).next()

notFriends = results['not-friends']
paths = results['paths']

def formatter(label, colors):
    if label in notFriends:
        colors.append('#11cc77')
    else:
        colors.append('yellow')

visualisation.plotPaths(paths, formatter)

## 2. Model Training (Entrenamiento modelo)

In [ ]:
from sagemaker import KMeans
from sagemaker import get_execution_role

role = get_execution_role()
print(role)

bucket = "alexbc.datalake"

data_location = 's3://{}/kidnapping-records/model/data'.format(bucket)
output_location = 's3://{}/kidnapping-records/model/output'.format(bucket)

print('training data will be uploaded to: {}'.format(data_location))
print('training artifacts will be uploaded to: {}'.format(output_location))

kmeans = KMeans(role=role,
                train_instance_count=1,
                train_instance_type='ml.m5.xlarge',
                output_path=output_location,
                k=10,
                data_location=data_location)

In [ ]:
crimeArray = crime.as_matrix().astype(np.float32)
crimeArray.shape
slice=crimeArray[:,1:5]

In [ ]:
%%time
kmeans.fit(kmeans.record_set(slice))

## 3. Model Deployment - (Mover el modelo a la producción).

In [ ]:
%%time
kmeans_predictor = kmeans.deploy(initial_instance_count=1,
                                 instance_type='ml.t2.large')

# 99. Other examples

https://aws.amazon.com/blogs/publicsector/estimating-hurricane-wind-speeds-with-machine-learning/